# 14. Crystal-Field Splitting, Electrochemistry, NMR/XPS & SciComm Templates

This tutorial walks through four high-impact chemistry, materials, and science-communication workflows:

1. **Crystal-Field $d$-Orbital Splitting** (`plot_crystal_field`): compare high-spin vs. low-spin filling across coordination geometries while conserving the $d$-orbital barycenter.
2. **Electrochemistry & Batteries** (`plot_cyclic_voltammetry`, `plot_tafel`, `plot_battery_profile`, `plot_dqdv`): visualize reversible redox peaks, Tafel overpotential kinetics, galvanostatic voltage plateaus, and differential capacity peaks.
3. **Analytical Spectroscopy** (`plot_nmr`, `plot_xps`): render first-order $^1\mathrm{H}$ NMR multiplet splitting trees with step integrals, plus XPS core-level deconvolution with iterative Shirley backgrounds.
4. **Turnkey SciComm Layout Templates** (`create_graphical_abstract`, `create_talk_slide`, `create_mechanism_comic`, `create_conference_poster`): build journal TOC abstracts, 16:9 presentation slides, 3-panel mechanism comics, and millimeter poster grids in one call.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import xkcd_matchem as xm
from xkcd_matchem.plots import crystalfield

## 1. Crystal-Field $d$-Orbital Splitting

When ligands surround a transition-metal ion, electrostatic repulsion lifts the five-fold degeneracy of its $d$ orbitals while conserving their weighted average energy (the **barycenter**, $\sum_i g_i E_i = 0$). In an octahedral ($O_h$) complex, orbitals pointing directly at the ligands ($e_g$: $d_{z^2}, d_{x^2-y^2}$) rise by $+0.6\,\Delta_o$, while orbitals nestled between ligands ($t_{2g}$: $d_{xy}, d_{xz}, d_{yz}$) drop by $-0.4\,\Delta_o$.

For $d^4$–$d^7$ ions, weak-field ligands produce **high-spin** configurations (spreading electrons singly across all five orbitals by Hund's rule before pairing), whereas strong-field ligands like $\mathrm{CN^-}$ favor **low-spin** pairing in the lower $t_{2g}$ shell. `plot_crystal_field()` supports `"octahedral"`, `"tetrahedral"`, `"square_planar"`, `"trigonal_bipyramidal"`, and custom level definitions.

In [ ]:
for geom in ("octahedral", "tetrahedral", "square_planar"):
    lvls = crystalfield.crystal_field_levels(geom, delta=1.5)
    bary = sum(deg * eng for _, eng, deg in lvls)
    print(f"{geom:14s} barycenter sum = {bary:+.1e} eV")

with xm.xkcd_style("classic"):
    fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.6))
    xm.plot_crystal_field(
        geometry="octahedral",
        d_electrons=6,
        spin_state="high",
        delta=1.25,
        ax=axes[0],
        title=r"High-Spin $\mathrm{Fe^{2+}}$ ($O_h$, $d^6$)",
    )
    xm.plot_crystal_field(
        geometry="octahedral",
        d_electrons=6,
        spin_state="low",
        delta=2.40,
        ax=axes[1],
        title=r"Low-Spin $\mathrm{[Fe(CN)_6]^{4-}}$ ($O_h$, $d^6$)",
    )
    xm.plot_crystal_field(
        geometry="square_planar",
        d_electrons=8,
        spin_state="low",
        delta=2.00,
        ax=axes[2],
        title=r"Square-Planar $\mathrm{Pt^{2+}}$ ($D_{4h}$, $d^8$)",
    )
    fig.tight_layout()
    plt.show()

## 2. Electrochemistry: Cyclic Voltammetry, Tafel Kinetics & Battery `dQ/dV`

- **Cyclic Voltammetry (`plot_cyclic_voltammetry`)** sweeps electrode potential forward and backward to probe electron-transfer reversibility. It automatically locates anodic ($E_{pa}$) and cathodic ($E_{pc}$) peaks, marks the half-wave potential $E_{1/2} = \tfrac{1}{2}(E_{pa} + E_{pc})$, reports peak separation $\Delta E_p$, and supports both `convention="iupac"` (positive potential right) and `convention="us"` (inverted potential axis).
- **Tafel Plots (`plot_tafel`)** fit the linear kinetic regime $\eta = a + b \log_{10}|j|$ to report the Tafel slope $b$ (in $\mathrm{mV/dec}$, revealing the rate-determining step) and exchange current density $\log_{10} j_0$.
- **Galvanostatic Cycling & Differential Capacity (`plot_battery_profile`, `plot_dqdv`)** pair voltage-vs.-capacity curves with their derivative $dQ/dV$: flat two-phase intercalation plateaus (like $\mathrm{LiFePO_4}$ near 3.4 V) sharpen into crisp redox peaks.

In [ ]:
# Synthesize a reversible Fe(CN)6 3-/4- voltammogram
e_fwd = np.linspace(0.00, 0.72, 160)
e_rev = np.linspace(0.72, 0.00, 160)
u_a = (e_fwd - 0.382) / 0.036
j_fwd = (
    0.10
    + 2.45 * np.exp(-0.5 * u_a**2) * (1.0 + 0.30 * np.tanh(u_a))
    + 0.78
    / (1.0 + np.exp(-18.0 * (e_fwd - 0.405)))
    / np.sqrt(1.0 + np.maximum(e_fwd - 0.39, 0.0) * 6.5)
)
u_c = (e_rev - 0.338) / 0.036
join_w = np.exp(-((0.72 - e_rev) / 0.055) ** 2)
j_rev_core = (
    -0.12
    - 2.40 * np.exp(-0.5 * u_c**2) * (1.0 - 0.30 * np.tanh(u_c))
    - 0.72
    / (1.0 + np.exp(18.0 * (e_rev - 0.315)))
    / np.sqrt(1.0 + np.maximum(0.33 - e_rev, 0.0) * 6.5)
)
j_rev = (1.0 - join_w) * j_rev_core + join_w * float(j_fwd[-1])
e_cv = np.concatenate([e_fwd, e_rev])
j_cv = np.concatenate([j_fwd, j_rev])

# Synthesize Tafel kinetics (true slope ~ 40 mV/dec)
log_j = np.linspace(-2.5, 1.5, 36)
j_tafel = 10.0**log_j
eta_mv = 40.0 * (log_j + 2.8) + 6.0 * np.maximum(log_j - 0.6, 0.0) ** 2

# Synthesize LiFePO4 galvanostatic charge/discharge curves
v_grid = np.linspace(2.65, 3.95, 220)
q_chg = 158.0 / (1.0 + np.exp(-(v_grid - 3.46) / 0.022)) + 3.2 * (
    v_grid - 2.65
)
q_chg = (q_chg - float(q_chg[0])) * (162.0 / float(q_chg[-1] - q_chg[0]))
q_dis_asc = 158.0 / (1.0 + np.exp(-(v_grid - 3.39) / 0.022)) + 3.2 * (
    v_grid - 2.65
)
q_dis_asc = (q_dis_asc - float(q_dis_asc[0])) * (
    162.0 / float(q_dis_asc[-1] - q_dis_asc[0])
)
q_dis, v_dis = q_dis_asc[::-1], v_grid[::-1]

with xm.xkcd_style("classic"):
    fig, axes = plt.subplots(2, 2, figsize=(12.4, 9.2))
    xm.plot_cyclic_voltammetry(
        e_cv,
        j_cv,
        convention="iupac",
        potential_label="Potential E (V vs. SHE)",
        ax=axes[0, 0],
        title=r"$\mathrm{[Fe(CN)_6]^{3-/4-}}$ Cyclic Voltammetry",
    )
    xm.plot_tafel(
        eta_mv,
        j_tafel,
        fit_range=(-2.2, 0.5),
        ax=axes[0, 1],
        title="HER Catalyst Tafel Kinetics",
    )
    xm.plot_battery_profile(
        q_dis,
        v_dis,
        charge_voltage=v_grid,
        charge_capacity=q_chg,
        plateau_shade=True,
        ax=axes[1, 0],
        title=r"$\mathrm{LiFePO_4}$ Charge / Discharge",
    )
    xm.plot_dqdv(
        v_dis,
        q_dis,
        charge_voltage=v_grid,
        charge_capacity=q_chg,
        smooth_window=5,
        annotate_peaks=1,
        ax=axes[1, 1],
        title=r"$\mathrm{LiFePO_4}$ Differential Capacity (dQ/dV)",
    )
    fig.tight_layout()
    plt.show()

## 3. $^1\mathrm{H}$ NMR Multiplet Trees & XPS Shirley Deconvolution

- **$^1\mathrm{H}$ NMR (`plot_nmr`)** uses the standard inverted chemical-shift scale $\delta\;(\mathrm{ppm})$ with tetramethylsilane (TMS) at 0 ppm. Given resonance chemical shifts, relative proton integrals, multiplicities (`"s"`, `"d"`, `"t"`, `"q"`, `"dd"`, ...), and scalar couplings $J$ (Hz), `plot_nmr()` synthesizes exact first-order Pascal splitting trees ($\Delta\delta = J / \nu_0$), Lorentzian lineshapes, and a cumulative step-integral curve.
- **Core-Level XPS (`plot_xps`)** plots intensity against an inverted Binding Energy (eV) axis, iteratively solves the inelastic **Shirley background** (proportional at each energy to the integrated peak area at lower binding energy), and overlays chemical-state component envelopes.

In [ ]:
et_oac_nmr = [
    {
        "shift": 4.12,
        "integral": 2.0,
        "multiplicity": "q",
        "j_hz": 7.1,
        "label": r"$\mathrm{-OCH_2-}$",
    },
    {
        "shift": 2.05,
        "integral": 3.0,
        "multiplicity": "s",
        "j_hz": 0.0,
        "label": r"$\mathrm{CH_3CO-}$",
    },
    {
        "shift": 1.26,
        "integral": 3.0,
        "multiplicity": "t",
        "j_hz": 7.1,
        "label": r"$\mathrm{-CH_3}$",
    },
]

be_grid = np.linspace(281.0, 292.5, 320)
xps_comps = [
    {"label": "C-C / C-H", "center": 284.8, "amplitude": 1.00, "fwhm": 1.15},
    {"label": "C-O", "center": 286.4, "amplitude": 0.46, "fwhm": 1.25},
    {"label": "O-C=O", "center": 288.9, "amplitude": 0.34, "fwhm": 1.25},
]
net_xps = np.zeros_like(be_grid)
for comp in xps_comps:
    net_xps += xm.broaden_peaks(
        [float(comp["center"])],
        [float(comp["amplitude"])],
        be_grid,
        fwhm=float(comp["fwhm"]),
        profile="pseudo_voigt",
        normalize="height",
    )
cum = np.cumsum(net_xps)
cum = (cum - cum[0]) / max(float(cum[-1] - cum[0]), 1e-12)
rng = np.random.default_rng(7)
raw_xps = 0.14 + 0.24 * cum + net_xps + rng.normal(0.0, 0.015, be_grid.size)

with xm.xkcd_style("classic"):
    fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.8))
    xm.plot_nmr(
        et_oac_nmr,
        spectrometer_mhz=400.0,
        ppm_range=(-0.3, 4.9),
        fwhm_ppm=0.016,
        show_integrals=True,
        show_trees=True,
        show_tms=True,
        ax=axes[0],
        title=r"Ethyl Acetate $^1\mathrm{H}$ NMR (400 MHz)",
    )
    xm.plot_xps(
        be_grid,
        raw_xps,
        components=xps_comps,
        background="shirley",
        invert_x=True,
        ax=axes[1],
        title="C 1s XPS (Shirley Deconvolution)",
    )
    fig.tight_layout()
    plt.show()

## 4. Turnkey SciComm Templates: Graphical Abstracts, Talk Slides & Mechanism Comics

When preparing a journal Table-of-Contents (TOC) graphic, seminar slide, conference poster, or outreach comic strip, you do not have to hand-tune subplot geometry from scratch. `xkcd-matchem` provides four turnkey layout recipes:

- `xm.create_graphical_abstract(...)` — millimeter-sized canvas with an optional central workflow arrow and bold ink takeaway strip.
- `xm.create_talk_slide(...)` — 16:9 widescreen slide (`"two_column"`, `"hero_right_notes"`, or `"three_panel"`) with an optional `xkcd.com` stick-scientist presenter.
- `xm.create_mechanism_comic(...)` — $N$-panel story strip with pre-positioned inset axes and non-overlapping characters.
- `xm.create_conference_poster(...)` — millimeter-accurate multi-card poster board (`"A0_landscape"`, `"A0_portrait"`, or `"compact_board"`).

In [ ]:
# A. Journal TOC / Graphical Abstract
ga = xm.create_graphical_abstract(
    title="Rutile TiO2 (110) -> Sabatier Activity",
    takeaway="Optimal *H binding energy sits right at the volcano apex",
    panels=2,
    size_mm=(136.0, 76.5),
    arrow_label="DFT",
    dpi=160,
)
xm.plot_coordination(
    "tio2_slab.poscar",
    "Ti",
    "O",
    supercell=(2, 1, 1),
    show_cell=False,
    ax=ga.left_ax,
    preset="graphical_abstract",
    title="TiO2 Octahedra",
)
if ga.right_ax is not None:
    xm.plot_volcano(
        [-0.75, -0.35, -0.05, 0.25, 0.65],
        [-2.9, -1.35, -0.35, -1.25, -3.1],
        labels=["Ru", "Rh", "Pt", "Pd", "Au"],
        ylabel="Limiting Potential (V)",
        ax=ga.right_ax,
        preset="graphical_abstract",
        title="HER Volcano",
    )
plt.show()

In [ ]:
# B. 16:9 Talk Slide with Stick-Scientist Presenter
bands = xm.load("si_qe_bands.dat")
slide = xm.create_talk_slide(
    title="Silicon Bands & Charged-Defect Levels",
    subtitle="Self-consistent DFT bands paired with defect thermodynamics",
    takeaway="Indirect 1.12 eV gap sets the p-type V_Si stability window",
    layout="two_column",
    preset="talk_16x9",
    character="ponytail",
    speech="Check V_Si (+2/0)!",
    dpi=140,
)
xm.plot_bands(
    bands,
    ax=slide.axes["left"],
    ylim=(-5.5, 4.5),
    preset="talk_16x9",
    title="Si Band Structure",
)
xm.plot_defect_formation(
    {
        "V_Si": [(2, 2.0), (1, 2.35), (0, 3.05)],
        "Si_i": [(1, 2.5), (0, 2.95), (-1, 3.85)],
    },
    band_gap=1.12,
    ax=slide.axes["right"],
    preset="talk_16x9",
    title="Si Defect Levels",
)
plt.show()

In [ ]:
# C. 3-Panel SN2 Mechanism Comic Strip + Compact Poster Check
neb = xm.load_trajectory("sn2_neb.xyz")
cam_sn2 = xm.Camera(azimuth=-115.0, elevation=20.0, perspective=0.25)
comic = xm.create_mechanism_comic(
    captions=(
        "1. Backside Approach",
        "2. Pentacoordinate TS!",
        "3. Walden Inversion",
    ),
    speeches=(
        "Cl- lines up...",
        "Look at that barrier!",
        "Inversion complete!",
    ),
    characters=("cueball", "ponytail", "goggles"),
    poses=("stand", "point", "cheer"),
    figsize=(13.6, 4.2),
)
xm.render_structure_3d(
    neb[0],
    ax=comic.inset_axes[0],
    camera=cam_sn2,
    show_cell=False,
)
xm.plot_neb(neb, ax=comic.inset_axes[1])
comic.inset_axes[1].set_ylim(-0.08, 1.16)
xm.render_structure_3d(
    neb[-1],
    ax=comic.inset_axes[2],
    camera=cam_sn2,
    show_cell=False,
)
plt.show()

poster = xm.create_conference_poster(
    title="xkcd-matchem Compact Poster Layout",
    authors="A. Researcher",
    affiliation="Materials SciComm Lab",
    size="compact_board",
)
print("Poster sections:", list(poster.cards.keys()))
plt.close(poster.board.figure)